# Assignment 2: Social Media & Network Analysis
## "The impact of AI on Music Sectors (Individual, Communities and Industries)"

Undergraduate (UG) Group 1
- Joshua Steinke (s4091863)
- Paul Venatt (s4089896)
- Putu Adhi Wiguna (s4097286)



In [1]:
# Import packages here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import re
import json
import nltk
import string
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from scripts.YouTubeProcessing import YouTubeProcessing


In [2]:
with open('./data/youtubeFlatComments_AI_on_music.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data['comments'])
df.head()

,video_id,video_title,channel_id,channel_title,video_published_at,video_view_count,video_like_count,video_comment_count,comment_id,parent_comment_id,comment_type,author_display_name,author_channel_id,text,text_original,like_count,published_at,updated_at,reply_to_author_display_name,reply_to_author_channel_id
0,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg,NaN,top_level,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw,Never could have predicted this crossover. Two...,Never could have predicted this crossover. Two...,1441,2026-05-13T15:32:19Z,2026-05-13T15:32:19Z,NaN,NaN
1,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhCufgK72,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@MidnightUnity,UCdisatKsUEaCeliNJ6BdKZg,"I was about to say two of my favourite nerds, ...","I was about to say two of my favourite nerds, ...",28,2026-05-13T15:42:20Z,2026-05-13T15:42:20Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw
2,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhSyhPznS,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@AdımızAdımız,UCRELhN7padwcwT94a0m0tnQ,fr,fr,1,2026-05-13T15:44:32Z,2026-05-13T15:44:32Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw
3,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhd5JSn27,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@mymasmith7848,UC7ps19AcOCVsJB3MzW_qsmQ,"Yeeeeehah, I am so eagerly the next two hours ...","Yeeeeehah, I am so eagerly the next two hours ...",2,2026-05-13T15:46:03Z,2026-05-13T15:46:03Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw
4,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkjpDEB6Po,Ugz_vLToQloKib1Dsmt4AaABAg,reply,@zlquis,UCbhxNik1HRXNuFjBb2nXMvg,Literally my 2 favourite content creators. I'm...,Literally my 2 favourite content creators. I'm...,9,2026-05-13T16:05:11Z,2026-05-13T16:05:11Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw


In [3]:
print(f'Shape: {df.shape}')
df.columns

Shape: (14342, 20)


Index(['video_id', 'video_title', 'channel_id', 'channel_title',
       'video_published_at', 'video_view_count', 'video_like_count',
       'video_comment_count', 'comment_id', 'parent_comment_id',
       'comment_type', 'author_display_name', 'author_channel_id', 'text',
       'text_original', 'like_count', 'published_at', 'updated_at',
       'reply_to_author_display_name', 'reply_to_author_channel_id'],
      dtype='str')

In [4]:
# Get a summary of the dataset
data_summary = {
    'number_of_videos': df['video_id'].nunique(),
    'number_of_comments_and_replies': len(df),
    'number_of_unique_users': df['author_channel_id'].nunique(),
    'number_of_top_level_comments': (df['comment_type'] == 'top_level').sum(),
    'number_of_replies': (df['comment_type'] == 'reply').sum()
}

display(data_summary)

{'number_of_videos': 25,
 'number_of_comments_and_replies': 14342,
 'number_of_unique_users': 9265,
 'number_of_top_level_comments': np.int64(2331),
 'number_of_replies': np.int64(12011)}

In [5]:
df['text'].isna().sum() + (df['text'].astype(str).str.strip() == '').sum()

np.int64(7)

In [6]:
df.isna().sum().sort_values(ascending=False)

reply_to_author_channel_id      2331
reply_to_author_display_name    2331
parent_comment_id               2331
author_display_name                0
updated_at                         0
published_at                       0
like_count                         0
text_original                      0
text                               0
author_channel_id                  0
video_id                           0
video_title                        0
comment_id                         0
video_comment_count                0
video_like_count                   0
video_view_count                   0
video_published_at                 0
channel_title                      0
channel_id                         0
comment_type                       0
dtype: int64

In [7]:
# Using NLTK stopwords
nltk.download('stopwords')
tokeniser = TweetTokenizer()

"""
Using English stopwords and punctuation,
as well as some additional stopwords that don't add much meaning to comments
"""
stopwords = stopwords.words('english') + list(string.punctuation)
additional_stopwords = ['ai', 'music', 'song', 'songs', 'video', 'like', 'just', 'get', 'would']
stopwords += (additional_stopwords)

stemmer = PorterStemmer()
processor = YouTubeProcessing(tokeniser=tokeniser, stopwords=stopwords, stemmer=stemmer)

[nltk_data] Downloading package stopwords to /Users/adhi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
"""
Apply text processing functions from YouTubeProcessing.py to:

1. Clean the raw text
2. Tokenise
3. Join the processed tokens back into a string
"""
df['cleaned_text'] = df['text'].apply(processor.clean_raw_text)
df['tokens'] = df['text'].apply(processor.process)
df['processed_text'] = df['tokens'].apply(processor.joined_tokens)

# Inspect the newly created columns
df[['text', 'cleaned_text', 'tokens', 'processed_text']].head()

,text,cleaned_text,tokens,processed_text
0,Never could have predicted this crossover. Two...,never could have predicted this crossover. two...,"[never, could, predict, crossov, two, favorit,...",never could predict crossov two favorit studi ...
1,"I was about to say two of my favourite nerds, ...","i was about to say two of my favourite nerds, ...","[say, two, favourit, nerd, studi, actual, nice...",say two favourit nerd studi actual nicer way w...
2,fr,fr,[fr],fr
3,"Yeeeeehah, I am so eagerly the next two hours ...","yeeeeehah, i am so eagerly the next two hours ...","[yeeeeehah, eagerli, next, two, hour, fine, li...",yeeeeehah eagerli next two hour fine listen
4,Literally my 2 favourite content creators. I'm...,literally my 2 favourite content creators. i'm...,"[liter, favourit, content, creator, even, go, ...",liter favourit content creator even go see ada...


In [9]:
df['text_length'] = df['cleaned_text'].str.len()
df['token_count'] = df['tokens'].apply(len)

df[['text_length', 'token_count']].describe()

,text_length,token_count
count,14342.000000,14342.000000
mean,173.134221,14.858039
std,282.165460,23.987252
min,0.000000,0.000000
25%,34.000000,3.000000
50%,78.500000,7.000000
75%,189.000000,16.000000
max,6192.000000,502.000000


In [10]:
# Remove very short comments
df = df[df['token_count'] >= 2]

In [11]:
# Save the cleaned comments dataset into a CSV and JSON file
df.to_csv('data/clean_comments.csv', index=False)
df.to_json('data/clean_comments.json', orient='records', force_ascii=False, indent=2)

In [12]:
reply_count = (df['comment_type'] == 'reply').sum()
top_level_count = (df['comment_type'] == 'top_level').sum()

print('Top-level comments:', top_level_count)
print('Replies:', reply_count)
print('Reply percentage:', round(reply_count / len(df) * 100, 2), '%')

Top-level comments: 2132
Replies: 10748
Reply percentage: 83.45 %


### 1. SNA Measures

In [13]:
data = pd.read_csv('./data/clean_comments.csv')
data.head()

,video_id,video_title,channel_id,channel_title,video_published_at,video_view_count,video_like_count,video_comment_count,comment_id,parent_comment_id,...,like_count,published_at,updated_at,reply_to_author_display_name,reply_to_author_channel_id,cleaned_text,tokens,processed_text,text_length,token_count
0,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg,NaN,...,1441,2026-05-13T15:32:19Z,2026-05-13T15:32:19Z,NaN,NaN,never could have predicted this crossover. two...,"['never', 'could', 'predict', 'crossov', 'two'...",never could predict crossov two favorit studi ...,79,8
1,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhCufgK72,Ugz_vLToQloKib1Dsmt4AaABAg,...,28,2026-05-13T15:42:20Z,2026-05-13T15:42:20Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw,"i was about to say two of my favourite nerds, ...","['say', 'two', 'favourit', 'nerd', 'studi', 'a...",say two favourit nerd studi actual nicer way w...,108,11
2,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkhd5JSn27,Ugz_vLToQloKib1Dsmt4AaABAg,...,2,2026-05-13T15:46:03Z,2026-05-13T15:46:03Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw,"yeeeeehah, i am so eagerly the next two hours ...","['yeeeeehah', 'eagerli', 'next', 'two', 'hour'...",yeeeeehah eagerli next two hour fine listen,63,7
3,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWkjpDEB6Po,Ugz_vLToQloKib1Dsmt4AaABAg,...,9,2026-05-13T16:05:11Z,2026-05-13T16:05:11Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw,literally my 2 favourite content creators. i'm...,"['liter', 'favourit', 'content', 'creator', 'e...",liter favourit content creator even go see ada...,88,11
4,Sbv0iX_EyLM,AI Music is Not Music - Adam Neely,UC7kIy8fZavEni8Gzl8NLjOQ,Alex O'Connor,2026-05-13T15:31:35Z,229348,8458,3861,Ugz_vLToQloKib1Dsmt4AaABAg.AWkg3VfSPEZAWklhIqRkuH,Ugz_vLToQloKib1Dsmt4AaABAg,...,8,2026-05-13T16:21:35Z,2026-05-13T16:21:35Z,@foxmocs6443,UCinA3pJ4_XChanp3nvpfNsw,"alex is a pretty good musician himself, so i'm...","['alex', 'pretti', 'good', 'musician', 'surpris']",alex pretti good musician surpri,60,5


In [14]:
# Create a dataset consiting of replies only
replies_data = data[
    (data['comment_type'] == 'reply') &
    (data['author_channel_id'].notna()) &
    (data['reply_to_author_channel_id'].notna())
].copy()

print('Number of reply rows:', len(replies_data))
print('Unique replying users:', replies_data['author_channel_id'].nunique())
print('Unique replied-to users:', replies_data['reply_to_author_channel_id'].nunique())

Number of reply rows: 10748
Unique replying users: 6519
Unique replied-to users: 986


In [15]:
# Remove user replies to themselves
replies_data = replies_data[replies_data['author_channel_id'] != replies_data['reply_to_author_channel_id']].copy()

In [16]:
# Creating the edge list
edge_data = (
    replies_data
    .groupby([
        'author_channel_id',
        'reply_to_author_channel_id'
    ])
    .agg(
        weight=('comment_id', 'count'),
        source_name=('author_display_name', 'first'),
        target_name=('reply_to_author_display_name', 'first')
    )
    .reset_index()
    .rename(columns={
        'author_channel_id': 'source',
        'reply_to_author_channel_id': 'target'
    })
)

edge_data.head()

,source,target,weight,source_name,target_name
0,UC-143ND7KXdjfcqMXbZH5Ng,UCZllv4L7S3t16kgLuN33NiA,1,@Tooruofficial1,@GramGramAnimations
1,UC-1yf1p7WRvY3z2L4MamGHg,UCgfZdkzvVUHf4vKPj-R4_7A,1,@Page-Humbuckers1973,@controlledburst
2,UC-3GMsrNch7u7837iLI26bA,UCPTa2aqVGv1bxCMlwkWOxYA,1,@vocecaiunocontodomalakoi7541,@ZBspicey
3,UC-3tABkXUfdRJFQ1Zx39MbA,UC9-tyozbTOKwJPZ6lEvAb4g,2,@jackweitzman6697,@foxyrand
4,UC-5zrhkAivIM0jKEWyLgVJg,UCxB219ufjxNVadCO3GPpotA,1,@AliBreee,@therealunclebbq


In [17]:
# Check the summary of the edge data
print('Number of unique directed edges:', len(edge_data))
print('Average edge weight:', edge_data['weight'].mean())
print('Maximum edge weight:', edge_data['weight'].max())

Number of unique directed edges: 7958
Average edge weight: 1.291153556169892
Maximum edge weight: 32


In [18]:
# Building the NetworkX graph
G = nx.from_pandas_edgelist(
    edge_data,
    source='source',
    target='target',
    edge_attr=['weight', 'source_name', 'target_name'],
    create_using=nx.DiGraph()
)

print('Number of nodes:', G.number_of_nodes())
print('Number of edges:', G.number_of_edges())
print('Network density:', nx.density(G))

Number of nodes: 7255
Number of edges: 7958
Network density: 0.00015121294328070522


In [19]:
# Calculate the centrality using 3 measures
in_degree = dict(G.in_degree(weight='weight'))
out_degree = dict(G.out_degree(weight='weight'))
pagerank = nx.pagerank(G, weight='weight')
betweenness = nx.betweenness_centrality(G, weight='weight', normalized=True)

In [20]:
# User lookup dictionary telling which ID belongs to which channel
usernames = {}
for _, row in replies_data.iterrows():
    usernames[row['author_channel_id']] = row['author_display_name']
    usernames[row['reply_to_author_channel_id']] = row['reply_to_author_display_name']

In [21]:
# Combining all centrality measures into a one DataFrame
centrality_data = pd.DataFrame({
    'id': list(G.nodes()),
    'username': [usernames.get(node, node) for node in G.nodes()],
    'weighted_in_degree': [in_degree.get(node, 0) for node in G.nodes()],
    'weighted_out_degree': [out_degree.get(node, 0) for node in G.nodes()],
    'pagerank': [pagerank.get(node, 0) for node in G.nodes()],
    'betweenness': [betweenness.get(node, 0) for node in G.nodes()]
})

centrality_data.head()

,id,username,weighted_in_degree,weighted_out_degree,pagerank,betweenness
0,UC-143ND7KXdjfcqMXbZH5Ng,@Tooruofficial1,0,1,0.000076,0.000000
1,UCZllv4L7S3t16kgLuN33NiA,@GramGramAnimations,97,1,0.006486,0.000004
2,UC-1yf1p7WRvY3z2L4MamGHg,@Page-Humbuckers1973,0,1,0.000076,0.000000
3,UCgfZdkzvVUHf4vKPj-R4_7A,@controlledburst,65,0,0.004602,0.000000
4,UC-3GMsrNch7u7837iLI26bA,@vocecaiunocontodomalakoi7541,0,1,0.000076,0.000000


In [22]:
centrality_data.sort_values('weighted_in_degree', ascending=False).head(10)

,id,username,weighted_in_degree,weighted_out_degree,pagerank,betweenness
552,UC4PIiYewI1YGyiZvgNlJNrA,@CharlesCornellStudios,100,4,0.004515,0.000016
66,UCAiYGga6xkRUnXYhaGd0WNA,@PuRpLe0MiSt,99,0,0.004738,0.000000
128,UCOrK28-rchatUNaXf8QgBPw,@halsti99,99,0,0.003174,0.000000
97,UCUJDTsUzE0s6xAwdNlwfXCA,@Prog-t9d,98,0,0.004221,0.000000
182,UC93SAoFOTG75UTTKa7CTCrw,@andriypredmyrskyy7791,97,0,0.002116,0.000000
1,UCZllv4L7S3t16kgLuN33NiA,@GramGramAnimations,97,1,0.006486,0.000004
242,UCQFtmcpgr7NkcvuaCCJdViA,@believershope,96,0,0.003732,0.000000
28,UCUhCiBjuc9YwfeYqQwQjlpA,@skirkstar,94,0,0.004125,0.000000
124,UCbpmPHT679KOANvgVKg1DzA,@3DThrills,93,0,0.003700,0.000000
393,UC8j3fdKGM2d-rv1qFp_QxLw,@cbobschloss,92,0,0.004284,0.000000


In [24]:
centrality_data.sort_values('weighted_out_degree', ascending=False).head(10)

,id,username,weighted_in_degree,weighted_out_degree,pagerank,betweenness
180,UCmeU2DYiVy80wMBGZzEWnbw,@PaulJLipsky,9,66,0.000610,0.000011
1868,UCf658Qpw_ou8oL_mityeP-A,@broughttoyouopmsoulrevival,2,60,0.000174,0.000002
3494,UCSh5zO3a_AZWmzukVmwvVJQ,@RusticRoadCountry,0,44,0.000076,0.000000
6387,UCrEJc2qvac8VfdVySj00t3w,@TevezNedro,0,40,0.000076,0.000000
7170,UCzFQwsVY0lcAOoorfABJ4_w,@EliMercerOfficial,0,37,0.000076,0.000000
4082,UCXkIMilviTtF_EFYC4ylOJg,@aahhhhhhhhhhhhh,0,35,0.000076,0.000000
6717,UCuajBd4igtzl5xjvKS9mYtw,@conspiracymusic,0,27,0.000076,0.000000
3038,UCOQfLCVu3VCrdwA_kqWtIEQ,@yuna17994,0,26,0.000076,0.000000
6308,UCqRNyvicCCqLIySuLK8DdIQ,@eightcoins4401,0,25,0.000076,0.000000
3316,UCRE-c6WE4RSZt1S9btINJ4w,@lukeshoo,0,23,0.000076,0.000000


In [26]:
centrality_data.sort_values('pagerank', ascending=False).head(10)

,id,username,weighted_in_degree,weighted_out_degree,pagerank,betweenness
1,UCZllv4L7S3t16kgLuN33NiA,@GramGramAnimations,97,1,0.006486,0.000004
415,UCRwX7nPX7kMbN8MEHSZG3ZQ,@Ellary_Rosewood,24,0,0.006247,0.000000
240,UCZJWUueXq2Gbvr8znMnBvzA,@sillybilly1489,83,0,0.005136,0.000000
41,UCKxT19NqLN5AYl4imxG2E5Q,@nathanbanks2354,86,0,0.004971,0.000000
66,UCAiYGga6xkRUnXYhaGd0WNA,@PuRpLe0MiSt,99,0,0.004738,0.000000
3,UCgfZdkzvVUHf4vKPj-R4_7A,@controlledburst,65,0,0.004602,0.000000
9,UCxB219ufjxNVadCO3GPpotA,@therealunclebbq,88,0,0.004590,0.000000
536,UCcqF79c07XaITCq1leWJlYQ,@Darksidedoom,74,0,0.004573,0.000000
552,UC4PIiYewI1YGyiZvgNlJNrA,@CharlesCornellStudios,100,4,0.004515,0.000016
177,UCOEkCtSMFWVm3en3618lzcA,@davisthedaviskids2574,79,0,0.004498,0.000000


In [27]:
centrality_data.sort_values('betweenness', ascending=False).head(10)

,id,username,weighted_in_degree,weighted_out_degree,pagerank,betweenness
552,UC4PIiYewI1YGyiZvgNlJNrA,@CharlesCornellStudios,100,4,0.004515,0.000016
777,UCfEJZmjVvEXt0RWURwP5Uag,@staticloop-o8r,29,9,0.002251,0.000015
326,UC7epYXoxGQlErIpBOYoCj9A,@AIAutomationLabs,40,20,0.002426,0.000013
180,UCmeU2DYiVy80wMBGZzEWnbw,@PaulJLipsky,9,66,0.000610,0.000011
7,UC9-tyozbTOKwJPZ6lEvAb4g,@foxyrand,37,1,0.003242,0.000006
1,UCZllv4L7S3t16kgLuN33NiA,@GramGramAnimations,97,1,0.006486,0.000004
120,UCA9ddiLbmBNN1pDlEGDKiuw,@MergoGreat1,64,8,0.002269,0.000004
1868,UCf658Qpw_ou8oL_mityeP-A,@broughttoyouopmsoulrevival,2,60,0.000174,0.000002
590,UCkmyqYWJ0x-nZjRMH6rI9tw,@NeoSoulReimahe,5,15,0.000402,0.000001
473,UCyGuZR2aP0RDpbXRhJycmRQ,@Burnrate,35,2,0.001584,0.000001


The user reply network was analysed using weighted in-degree, weighted out-degree, PageRank, and betweenness centrality. Weighted in-degree identifies users who received many replies, while weighted out-degree identifies users who actively replied to others. PageRank was used to identify structurally influential users, and betweenness centrality was used to identify users who acted as bridges between different parts of the discussion network.

In [29]:
centrality_data.to_csv('./data/replies_centrality.csv', index=False)

### References
